# Genre Prediction — Full Pipeline

**Model:** Binary Relevance with BernoulliNB (one classifier per genre)  
**Constraint:** No TF-IDF, no embedding models  
**Task:** Multi-label genre classification

---
## Roadmap
1. Load & audit raw data
2. Clean data (`DataCleaner`)
3. Train/test split
4. Feature engineering (`FeatureEngineer`) — fit on train only
5. Model training (`BernoulliNB` via `MultiOutputClassifier`)
6. Evaluation (per-genre F1, micro/macro F1)


---
## Phase 1 — Load & Audit

In [1]:
from utils import *
from utils_one import *

multi = False

data_folder = 'data'

In [2]:
# Load all CSV files into separate DataFrames for inspection
print("=" * 50)
print("Loading all CSV files separately:")
print("=" * 50)
try:
    data_dict = load_all_csv_files(data_folder)
    for name, df in data_dict.items():
        print(f"\n{name}:")
        print(df.head())
except FileNotFoundError as e:
    print(f"Error: {e}")

Loading all CSV files separately:
Loaded merged_dataset.csv: 1500 rows, 9 columns
Loaded TMDB IMDB Movies Dataset.csv: 435435 rows, 29 columns

merged_dataset:
                                                name  year movie_rated  \
0                                          Inception  2010       PG-13   
1  The Lord of the Rings: The Fellowship of the Ring  2001       PG-13   
2      The Lord of the Rings: The Return of the King  2003       PG-13   
3                              The Dark Knight Rises  2012       PG-13   
4              The Lord of the Rings: The Two Towers  2002       PG-13   

  run_length                       genres            release_date  rating  \
0   2h 28min  Action; Adventure; Sci-Fi;       16 July 2010 (USA)     8.8   
1   2h 58min   Action; Adventure; Drama;   19 December 2001 (USA)     8.8   
2   3h 21min  Adventure; Drama; Fantasy;   17 December 2003 (USA)     8.9   
3   2h 44min          Action; Adventure;       20 July 2012 (USA)     8.4   
4   2h 59m

In [3]:
# Audit each file before concatenation
for name, df in data_dict.items():
    print(f"\n--- Health Audit: {name} ---")
    cleaner = DataCleaner(df, verbose=False)
    health_report = cleaner.audit_column_health()
    print(health_report)


--- Health Audit: merged_dataset ---
         column     type  missing_or_nan  total_rows
0          name      str               0        1500
1          year    int64               0        1500
2   movie_rated      str               0        1500
3    run_length      str               0        1500
4        genres      str               0        1500
5  release_date      str               0        1500
6        rating  float64               0        1500
7    num_raters    int64               0        1500
8   num_reviews    int64               0        1500

--- Health Audit: TMDB IMDB Movies Dataset ---
                  column     type  missing_or_nan  total_rows
0                     id    int64               0      435435
1                  title      str               0      435435
2           vote_average  float64               0      435435
3             vote_count    int64               0      435435
4                 status      str               0      435435
5           

In [4]:
# Concatenate all CSV files with column mapping
print("\n" + "=" * 50)
print("Concatenating all CSV files with column mapping:")
print("=" * 50)

column_mappings = {
    'Movies': {
        'original_title': 'title',
        'vote_average': 'averageRating',
        'vote_count': 'numVotes',
    },
    'merged_dataset': {
        'name': 'title',
        'rating': 'averageRating',
        'num_raters': 'numVotes',
        'run_length': 'runtime',
    }
}
file_order = ['TMDB IMDB Movies Dataset', 'merged_dataset']

df = concatenate_all_files(
    'data',
    '*.csv',
    column_mappings=column_mappings,
    file_order=file_order
)

print(f"\nTotal rows: {len(df)}")
df.head()


Concatenating all CSV files with column mapping:
Processing 2 files in order:
  1. TMDB IMDB Movies Dataset
  2. merged_dataset

[BASE] Loading TMDB IMDB Movies Dataset.csv...
  Shape: (435435, 29)
  Columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'tconst', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'directors', 'writers', 'averageRating', 'numVotes', 'cast']

[merged_dataset] Loading merged_dataset.csv...
  Shape: (1500, 9)
  Columns: ['name', 'year', 'movie_rated', 'run_length', 'genres', 'release_date', 'rating', 'num_raters', 'num_reviews']
  Applying column mappings: {'name': 'title', 'rating': 'averageRating', 'num_raters': 'numVotes', 'run_length': 'runtime'}
  New columns found: ['year', 'num_reviews', 'movie_rated']
  Missing columns (will

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,year,num_reviews,movie_rated
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2784846,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",<NA>,<NA>,<NA>
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2476729,"Matthew McConaughey, Anne Hathaway, Michael Ca...",<NA>,<NA>,<NA>
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3132056,"Christian Bale, Heath Ledger, Aaron Eckhart, M...",<NA>,<NA>,<NA>
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1484844,"Sam Worthington, Zoe Saldaña, Sigourney Weaver...",<NA>,<NA>,<NA>
4,24428,The Avengers,7.71,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1546016,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ...",<NA>,<NA>,<NA>


---
## Phase 2 — Clean Data

In [5]:
cleaner = DataCleaner(df, verbose=True)

# Step 1: Fix column names first
cleaner.standardise_column_names()

# Step 2: Drop columns with no predictive value (ids, image paths, etc.)
cleaner.drop_useless_columns()

# Step 3: Drop highly empty columns (>80% missing)
cleaner.drop_highly_empty_columns(threshold=0.8, empty_values=['unset', 'unknown'])

# Step 4: Standardise all string data (lowercase, strip, unify NaNs)
cleaner.standardise_data()

# Step 5: Drop rows with no genre — these cannot be used for training
cleaner.drop_empty_genres()

# Step 6: Convert numeric and boolean columns
cleaner.standardise_numeric()
cleaner.standardise_boolean()

# Step 7: Normalise delimiters in list-like columns
cleaner.convert_strings_to_lists()

final_df = cleaner.current_df
print(f"\nFinal cleaned dataset shape: {final_df.shape}")
final_df.head()


-> STARTING COLUMN NAME STANDARDISATION
✓ All column names were already standard. No changes made.

✓ Dropped 6 useless columns: ['id', 'backdrop_path', 'poster_path', 'homepage', 'tconst', 'original_title']
-> Dropped 'year': 99.7% empty (435435/436935)
-> Dropped 'num_reviews': 99.7% empty (435435/436935)
-> Dropped 'movie_rated': 99.7% empty (435435/436935)

Dropped 3 null columns in total.
✓ Standardised 21 columns (lowercase, stripped, null-unified)
✓ Dropped 79148 rows with empty 'genres'
-> Converting to numeric: 'vote_average'
-> Converting to numeric: 'vote_count'
-> Converting to numeric: 'revenue'
-> Converting to numeric: 'runtime'
-> Converting to numeric: 'budget'
-> Converting to numeric: 'popularity'
-> Converting to numeric: 'averageRating'
-> Converting to numeric: 'numVotes'
✓ Standardised 10 numeric columns
-> Converting to boolean: 'adult'
✓ Standardised 1 boolean columns
-> Normalising order-sensitive list: 'production_companies'
-> Normalising order-sensitive li

,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,original_language,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,inception,8.364,34495.0,released,2010-07-15,8.255328e+08,148.0,False,160000000.0,en,...,"action, science fiction, adventure","legendary pictures, syncopy, warner bros. pict...","united kingdom, united states of america","english, french, japanese, swahili","rescue, mission, dream, airplane, paris, franc...",christopher nolan,christopher nolan,8.8,2784846.0,"leonardo dicaprio, joseph gordon-levitt, ken w..."
1,interstellar,8.417,32571.0,released,2014-11-05,7.017292e+08,169.0,False,165000000.0,en,...,"adventure, drama, science fiction","legendary pictures, syncopy, lynda obst produc...","united kingdom, united states of america",english,"rescue, future, spacecraft, race against time,...",christopher nolan,"jonathan nolan, christopher nolan",8.7,2476729.0,"matthew mcconaughey, anne hathaway, michael ca..."
2,the dark knight,8.512,30619.0,released,2008-07-16,1.004558e+09,152.0,False,185000000.0,en,...,"drama, action, crime, thriller","dc comics, legendary pictures, syncopy, isobel...","united kingdom, united states of america","english, mandarin","joker, sadism, chaos, secret identity, crime f...",christopher nolan,"jonathan nolan, christopher nolan, david s. go...",9.1,3132056.0,"christian bale, heath ledger, aaron eckhart, m..."
3,avatar,7.573,29815.0,released,2009-12-15,2.923706e+09,162.0,False,237000000.0,en,...,"action, adventure, fantasy, science fiction","dune entertainment, lightstorm entertainment, ...","united states of america, united kingdom","english, spanish","future, society, culture clash, space travel, ...",james cameron,james cameron,7.9,1484844.0,"sam worthington, zoe saldaña, sigourney weaver..."
4,the avengers,7.710,29166.0,released,2012-04-25,1.518816e+09,143.0,False,220000000.0,en,...,"science fiction, action, adventure",marvel studios,united states of america,"english, hindi, russian","new york city, superhero, shield, based on com...",joss whedon,"joss whedon, zak penn",8.0,1546016.0,"robert downey jr., chris evans, mark ruffalo, ..."


In [6]:
# Optional: inspect fuzzy duplicates in categorical columns
cleaner.find_fuzzy_duplicates()

""


In [7]:
# Optional: profile unique values after cleaning
display(cleaner.profile_unique_values())

,column,type,unique_count,analysis_type,detail,example
0,title,str,305901,Categorical,Textual categories,inception
1,vote_average,float64,4992,Numeric,Min: 0.0 | Max: 10.0,8.364
2,vote_count,float64,3598,Numeric,Min: 0.0 | Max: 34495.0,34495.0
3,status,str,6,Categorical,Textual categories,released
4,release_date,str,39116,Categorical,Textual categories,2010-07-15
5,revenue,float64,13476,Numeric,Min: 0.0 | Max: 2923706026.0,825532764.0
6,runtime,float64,512,Numeric,Min: 0.0 | Max: 14400.0,148.0
7,adult,bool,2,Numeric,Min: False | Max: True,False
8,budget,float64,3871,Numeric,Min: 0.0 | Max: 888000000.0,160000000.0
9,original_language,str,155,Categorical,Textual categories,en


In [8]:
# Inspect the genres column before binarization
print("Sample genre values:")
print(final_df['genres'].dropna().head(20).tolist())

print("\nGenre value counts (top 20 individual genres):")
from collections import Counter
all_genres = []
for val in final_df['genres'].dropna():
    all_genres.extend([g.strip() for g in str(val).split(',') if g.strip()])
print(Counter(all_genres).most_common(20))

Sample genre values:
['action, science fiction, adventure', 'adventure, drama, science fiction', 'drama, action, crime, thriller', 'action, adventure, fantasy, science fiction', 'science fiction, action, adventure', 'action, adventure, comedy', 'adventure, action, science fiction', 'drama', 'action, science fiction, adventure', 'thriller, crime', 'comedy, drama, romance', 'adventure, fantasy', 'action, science fiction, adventure', 'drama, western', 'drama, crime', 'adventure, science fiction, action', 'action, science fiction', 'drama, romance', 'crime, thriller, drama', 'adventure, fantasy, action']

Genre value counts (top 20 individual genres):
[('drama', 139545), ('comedy', 91361), ('documentary', 59345), ('romance', 35919), ('thriller', 33045), ('action', 31498), ('horror', 30363), ('animation', 25580), ('crime', 24590), ('tv movie', 18012), ('family', 17641), ('adventure', 16718), ('music', 16475), ('science fiction', 13762), ('fantasy', 13512), ('mystery', 12728), ('history', 10

---
## Phase 3 — Train / Test Split

We split BEFORE feature engineering to avoid data leakage.
The `FeatureEngineer` will be fitted only on train data.

In [9]:
from sklearn.model_selection import train_test_split

# Simple random split — 80% train, 20% test
# Note: proper multi-label stratification requires the 'iterative-stratification'
# library (pip install iterative-stratification). For now we use a random split,
# which is fine for most dataset sizes.
train_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42)

print(f"Train size: {len(train_df)} rows")
print(f"Test size:  {len(test_df)} rows")

# Reset index so concat and iloc operations are consistent
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

Train size: 286229 rows
Test size:  71558 rows


---
## Phase 4 — Feature Engineering

**Critical rule:** `fit()` is called only on train data.  
`transform()` is then applied to both train and test separately.  
This prevents any test-data information from leaking into the encoders.

In [10]:
engineer = FeatureEngineer(
    top_n_per_list_col=50,   # keep top-50 most frequent values per list column
    n_bins=5,                # bin numeric columns into 5 quantile buckets
    verbose=True
)

# Fit ONLY on training data
X_train, y_train = engineer.fit_transform(train_df)

# Transform test data using the SAME fitted parameters
X_test, y_test = engineer.transform(test_df)

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")


FITTING FEATURE ENGINEER ON TRAINING DATA

→ Fitting primary genre encoder...
  genre_1: 197 classes — ['action', 'action; adventure;', 'action; adventure; biography;', 'action; adventure; comedy;', 'action; adventure; crime;', 'action; adventure; drama;', 'action; adventure; fantasy;', 'action; adventure; history;', 'action; adventure; horror;', 'action; adventure; mystery;', 'action; adventure; sci-fi;', 'action; adventure; thriller;', 'action; adventure; war;', 'action; biography; comedy;', 'action; biography; drama;', 'action; comedy; crime;', 'action; comedy; fantasy;', 'action; comedy; music;', 'action; comedy; war;', 'action; crime; drama;', 'action; crime; fantasy;', 'action; crime; horror;', 'action; crime; mystery;', 'action; crime; sci-fi;', 'action; crime; thriller;', 'action; drama; history;', 'action; drama; music;', 'action; drama; mystery;', 'action; drama; sci-fi;', 'action; drama; thriller;', 'action; drama; war;', 'action; fantasy;', 'action; fantasy; horror;', 'act

In [12]:
# Inspect feature names
print(f"Total features: {X_train.shape[1]}")
print("\nFeature columns (first 40):")
print(X_train.columns[:40].tolist())

if multi:
    print("\nTarget genres:")
    print(y_train.columns.tolist())
else:
    print("\nTarget genres:")
    print(y_train.tolist())

Total features: 604

Feature columns (first 40):
['runtime_bin0', 'runtime_bin1', 'runtime_bin2', 'runtime_bin3', 'runtime_bin4', 'runtime_missing', 'runtime_runtime_bin0', 'runtime_runtime_bin1', 'runtime_runtime_bin2', 'runtime_runtime_bin3', 'runtime_runtime_bin4', 'budget_bin0', 'budget_budget_bin0', 'budget_missing', 'revenue_bin0', 'revenue_missing', 'revenue_revenue_bin0', 'popularity_bin0', 'popularity_bin1', 'popularity_bin2', 'popularity_bin3', 'popularity_bin4', 'popularity_missing', 'popularity_popularity_bin0', 'popularity_popularity_bin1', 'popularity_popularity_bin2', 'popularity_popularity_bin3', 'popularity_popularity_bin4', 'vote_average_bin0', 'vote_average_bin1', 'vote_average_bin2', 'vote_average_bin3', 'vote_average_missing', 'vote_average_vote_average_bin0', 'vote_average_vote_average_bin1', 'vote_average_vote_average_bin2', 'vote_average_vote_average_bin3', 'vote_count_bin0', 'vote_count_bin1', 'vote_count_bin2']

Target genres:
[173, 167, 90, 65, 125, 124, 125,

In [13]:
# Sanity check: all features should be binary (0 or 1)
unique_vals = X_train.apply(lambda col: col.unique())
non_binary = [col for col in X_train.columns if not set(X_train[col].unique()).issubset({0, 1})]
if non_binary:
    print(f"⚠️  Non-binary columns found (should not happen): {non_binary}")
else:
    print("✓ All feature columns are binary (0/1) — ready for BernoulliNB")

✓ All feature columns are binary (0/1) — ready for BernoulliNB


In [14]:
if multi:
    # Check genre distribution in training set
    print("Genre label distribution in training set (% of movies with each genre):")
    # genre_freq = y_train.mean().sort_values(ascending=False)
    # display(genre_freq.to_frame(name='frequency').style.format('{:.1%}'))

    genre_classes = engineer.get_genre_classes()
    for slot, classes in genre_classes.items():
        print(f"{slot}: {len(classes)} classes — {list(classes)}")

else:
    print("Primary genre distribution (top 20):")
    primary = final_df['genres'].apply(lambda x: str(x).split(',')[0].strip())
    print(primary.value_counts().head(20))

Primary genre distribution (top 20):
genres
drama              96116
comedy             63444
documentary        55129
animation          20591
action             20561
horror             20084
thriller           11486
romance            11043
crime              10906
music               7654
adventure           6837
family              5928
science fiction     5065
tv movie            4650
western             4190
fantasy             4073
mystery             3442
war                 2556
history             2532
comedy;               85
Name: count, dtype: int64


---
## Phase 5 — Model Training

Binary Relevance: one `BernoulliNB` classifier per genre, wrapped by `MultiOutputClassifier`.

In [15]:
# Train the model
# alpha: Laplace smoothing — higher values smooth more aggressively
# alpha=1.0 is the standard default; you can tune this
model = train_model(X_train, y_train, alpha=1.0)

print(f"✓ Model trained: {len(model.estimators_)} BernoulliNB classifiers")

AttributeError: 'BernoulliNB' object has no attribute 'estimators_'

---
## Phase 6 — Evaluation

In [ ]:
slot_reports = evaluate_model(model, X_test, y_test, engineer)


  genre_1: accuracy=0.3562  macro_F1=0.0379  weighted_F1=0.3446
  genre_2: accuracy=0.4422  macro_F1=0.1425  weighted_F1=0.4227
  genre_3: accuracy=0.6937  macro_F1=0.1052  weighted_F1=0.7167

  Exact match  (all 3 slots correct): 0.1765
  Partial match (≥1 genre correct):   0.5096



In [ ]:
if multi:
    for slot, report_df in slot_reports.items():
        print(f"\n--- {slot} ---")
        display(report_df.style.format({'precision': '{:.3f}', 'recall': '{:.3f}', 'f1-score': '{:.3f}'}))
    
else:
    display(
        slot_reports
        .style.format({'precision': '{:.3f}', 'recall': '{:.3f}', 'f1-score': '{:.3f}', 'support': '{:.0f}'})
        .background_gradient(subset=['f1-score'], cmap='RdYlGn')
    )


--- genre_1 ---


,precision,recall,f1-score,support
action,0.301,0.329,0.314,4131.000000
action; adventure; comedy;,0.000,0.000,0.000,2.000000
action; adventure; drama;,0.000,0.000,0.000,1.000000
action; adventure; fantasy;,0.000,0.000,0.000,8.000000
action; adventure; horror;,0.000,0.000,0.000,2.000000
action; adventure; mystery;,0.000,0.000,0.000,3.000000
action; adventure; sci-fi;,0.100,0.579,0.171,19.000000
action; adventure; thriller;,0.000,0.000,0.000,5.000000
action; comedy; crime;,0.000,0.000,0.000,1.000000
action; comedy; fantasy;,0.000,0.000,0.000,1.000000



--- genre_2 ---


,precision,recall,f1-score,support
action,0.114,0.127,0.120,1585.000000
adventure,0.140,0.031,0.050,1369.000000
animation,0.112,0.233,0.151,711.000000
comedy,0.466,0.058,0.103,4441.000000
crime,0.160,0.058,0.085,1799.000000
documentary,0.077,0.129,0.096,791.000000
drama,0.178,0.130,0.150,6892.000000
family,0.128,0.064,0.085,1461.000000
fantasy,0.117,0.020,0.034,1047.000000
history,0.153,0.127,0.139,1106.000000



--- genre_3 ---


,precision,recall,f1-score,support
action,0.048,0.053,0.050,472.000000
adventure,0.023,0.014,0.018,490.000000
animation,0.057,0.124,0.078,201.000000
comedy,0.054,0.005,0.010,944.000000
crime,0.086,0.049,0.062,698.000000
documentary,0.000,0.000,0.000,67.000000
drama,0.067,0.023,0.034,1624.000000
family,0.135,0.163,0.148,661.000000
fantasy,0.019,0.008,0.011,610.000000
history,0.040,0.102,0.057,325.000000
